# Learned GNN Physics Simulator for Particle Jet Tagging
**CS6012: Code-Based Project** | IIT Bombay  
**Group 11:** Rahul Kumar (25M2015) & Pradipta Sadhukhan (25D1080)  
**Supervisor:** Prof. Sharat Chandran  
**Reference Paper:** *Graph Neural Networks in Particle Physics* (arXiv:2007.13681)  

---
### Full Official 100,000-Jet Pythia 8.2 Collider Benchmark on Tesla T4 GPU:
This notebook trains both architectures on the **full 100,000 jets** of the official **Pythia 8.2 Quark-Gluon Collider Dataset** from Zenodo (Komiske et al., DOI: 10.5281/zenodo.3164691):
1. **Downloads and preprocesses** all 100,000 real LHC collision jets (50k Quarks + 50k Gluons).
2. **Trains Deep Sets Baseline** on T4 GPU (~1.5 minutes).
3. **Trains Dynamic EdgeConv GNN (ParticleNet-lite)** on T4 GPU (~6 to 8 minutes).
4. **Evaluates both models** on held-out test data, generating publication ROC curves.
5. **Updates pre-trained weights and demo slice** for seamless TA evaluation.

In [ ]:
# Cell 1: Verify Hardware & GPU
import torch

print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 60)

In [ ]:
# Cell 2: Setup Workspace Directory & Clone if needed (Colab-Friendly)
import os, sys

# 1. If running in fresh Google Colab, clone the repo if not present
if not os.path.exists("src") and not os.path.exists("../src") and not os.path.exists("/content/gnn_jet_tagging/src"):
    print("[*] Fresh Colab environment detected. Cloning repository...")
    !git clone https://github.com/KraaK3N/gnn_jet_tagging.git

# 2. Enter repository directory
if os.path.exists("/content/gnn_jet_tagging"):
    %cd /content/gnn_jet_tagging
elif os.path.exists("gnn_jet_tagging"):
    %cd gnn_jet_tagging
elif os.path.exists("../src"):
    %cd ..

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"[✓] Active Working Directory: {os.getcwd()}")
print(f"[✓] Verified 'src' module: {os.path.exists('src')}")

In [ ]:
# Cell 3: Fetch & Preprocess Full Official 100,000-Jet Pythia 8.2 Dataset
# Loads QG_jets.npz (106 MB) and converts all 100,000 jets into constituent 7-feature arrays
from src.data.pythia import prepare_pythia_dataset

prepare_pythia_dataset(
    output_path="data/pythia_100k.npz",
    sample_output_path="data/sample_jets.npz",
    num_jets=100000,
    seed=42
)

In [ ]:
# Cell 4: Train Deep Sets Baseline on Full 100,000 Real Pythia Jets (T4 GPU)
from src.utils.config import load_config
from src.training.train import run_training

cfg_baseline = load_config("configs/baseline.yaml")
cfg_baseline["data_path"] = "data/pythia_100k.npz"
cfg_baseline["training"]["epochs"] = 15
cfg_baseline["training"]["batch_size"] = 128

print("[*] Training Deep Sets on 100,000 Real Pythia Jets (GPU)...")
run_training(cfg_baseline)

In [ ]:
# Cell 5: Train Dynamic EdgeConv GNN (ParticleNet-lite) on Full 100,000 Real Pythia Jets (T4 GPU)
cfg_gnn = load_config("configs/gnn.yaml")
cfg_gnn["data_path"] = "data/pythia_100k.npz"
cfg_gnn["training"]["epochs"] = 15
cfg_gnn["training"]["batch_size"] = 128

print("[*] Training Dynamic EdgeConv GNN on 100,000 Real Pythia Jets (GPU)...")
run_training(cfg_gnn)

In [ ]:
# Cell 6: Comparative Evaluation & Visualizing Physical ROC Curves
from src.evaluation.evaluate import run_comparative_evaluation
from IPython.display import Image, display

run_comparative_evaluation(
    checkpoints=["checkpoints/deepsets_best.pt", "checkpoints/edgeconv_best.pt"],
    data_path="data/pythia_100k.npz",
    output_json="results/metrics_summary.json",
    output_roc="results/figures/roc_curve.png"
)

print("\n[*] Displaying True Real-World Pythia ROC Curve:")
display(Image(filename="results/figures/roc_curve.png"))

In [ ]:
# Cell 7: Verify Canned Demo for Teaching Assistants
# Tests that the TA evaluation runs in under 10 seconds on the new real Pythia slice!
!./run_demo.sh